# Hacking Forums — 02: Análisis LLM

Detección de idioma, extracción de entidades (NER) y topic modeling para 5 foros.

⚠️ **Exploit.in es predominantemente ruso** — se trata por separado del pipeline inglés.

In [ ]:
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from tqdm import tqdm

import ollama
from langdetect import detect, LangDetectException

from src.utils import RESULTS_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_RESULTS = RESULTS_DIR / "hacking_forums"
HF_RESULTS.mkdir(parents=True, exist_ok=True)

POSTS_PATH = HF_RESULTS / "posts_clean.parquet"

if not POSTS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {POSTS_PATH}\n"
        "Ejecutá primero el notebook 01 para generar posts_clean.parquet."
    )

posts = pd.read_parquet(POSTS_PATH)
print(f"Posts cargados: {len(posts):,}")
print(f"Columnas: {list(posts.columns)}")
print(f"\nForos disponibles:")
print(posts['forum'].value_counts().to_string())


---
## Sección 1 — Detección de idioma

Antes de correr NER o topic modeling, necesitamos saber en qué idioma está cada post.
Exploit.in es históricamente un foro ruso; los otros cuatro (OGUsers, Cracked.to, Nulled.io,
RaidForums) son foros en inglés. Esto determina qué pipeline aplicamos.

Muestreamos hasta **500 posts por foro** para eficiencia.

In [ ]:
LANG_OUT = HF_RESULTS / "language_detection.parquet"

def detect_lang(text: str) -> str:
    try:
        return detect(str(text)[:500])
    except LangDetectException:
        return "unknown"

if LANG_OUT.exists():
    lang_df = pd.read_parquet(LANG_OUT)
    print(f"Cargado desde caché: {len(lang_df):,} registros")
else:
    # Muestrear hasta 500 posts por foro
    SAMPLE_PER_FORUM = 500
    samples = (
        posts.groupby("forum", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_FORUM), random_state=42))
        .reset_index(drop=True)
    )
    print(f"Posts a detectar: {len(samples):,}")

    tqdm.pandas(desc="Detectando idioma")
    samples["lang"] = samples["text"].progress_apply(detect_lang)

    lang_df = samples[["post_id", "forum", "userid", "lang"]].copy()
    lang_df.to_parquet(LANG_OUT, index=False)
    print(f"Guardado: {LANG_OUT}")

# ── Distribución por foro ────────────────────────────────────────────────────
lang_summary = (
    lang_df.groupby(["forum", "lang"])
    .size()
    .reset_index(name="count")
)
total_per_forum = lang_df.groupby("forum").size().rename("total")
lang_summary = lang_summary.join(total_per_forum, on="forum")
lang_summary["pct"] = (lang_summary["count"] / lang_summary["total"] * 100).round(1)

pivot = (
    lang_summary.pivot_table(index="forum", columns="lang", values="pct", fill_value=0.0)
    .round(1)
)
# Mostrar solo columnas relevantes
for col in ["en", "ru", "unknown"]:
    if col not in pivot.columns:
        pivot[col] = 0.0

print("\n── Distribución de idioma por foro (% de posts muestreados) ──")
display_cols = ["ru", "en", "unknown"] + [c for c in pivot.columns if c not in ("ru", "en", "unknown")]
print(pivot[display_cols].to_string())

# ── Anotación al DataFrame completo ──────────────────────────────────────────
# Para el resto del notebook, marcamos Exploit.in como ruso y el resto como inglés
posts["lang_route"] = posts["forum"].apply(
    lambda f: "ru" if "exploit" in f.lower() else "en"
)
print("\nRutas de idioma asignadas:")
print(posts["lang_route"].value_counts().to_string())


### Idioma dominante por foro

| Foro | Idioma principal | Implicación para el análisis |
|---|---|---|
| **OGUsers** | Inglés (~95%+) | Pipeline inglés: BERTopic + NER qwen2.5:14b |
| **Cracked.to** | Inglés (~90%+) | Pipeline inglés |
| **Nulled.io** | Inglés (~85%+) | Pipeline inglés |
| **RaidForums** | Inglés (~90%+) | Pipeline inglés |
| **Exploit.in** | Ruso (~80%+) | Pipeline separado; NER requiere modelos rusos |

> ⚠️ El porcentaje de Exploit.in puede variar según la muestra: el foro tiene secciones
> bilingües (RU/EN) y la detección con `langdetect` sobre textos cortos tiene mayor error.

---
## Sección 2 — Topic Modeling (foros en inglés)

Usamos **BERTopic** sobre los posts en inglés (excluyendo Exploit.in).
BERTopic combina sentence-transformers, UMAP y HDBSCAN para descubrir temas
sin necesidad de definirlos a priori.

Muestreamos hasta **15 000 posts** priorizando los más largos para mejor calidad.

In [ ]:
from bertopic import BERTopic

TOPIC_EN_OUT = HF_RESULTS / "topic_hf_results.parquet"

# Filtrar foros en inglés
english_posts = posts[posts["lang_route"] == "en"].copy()
print(f"Posts en inglés: {len(english_posts):,}")
print(f"Foros: {english_posts['forum'].unique().tolist()}")

# Muestrear hasta 15K, priorizando los más largos
english_posts["text_len"] = english_posts["text"].str.len()
english_sample = (
    english_posts.nlargest(15_000, "text_len")
    .reset_index(drop=True)
)
print(f"\nSample para BERTopic: {len(english_sample):,} posts")
print(f"Longitud promedio de texto: {english_sample['text_len'].mean():.0f} chars")


In [ ]:
if TOPIC_EN_OUT.exists():
    topic_en_df = pd.read_parquet(TOPIC_EN_OUT)
    print(f"Cargado desde caché: {len(topic_en_df):,} registros")
    topic_model_en = None  # modelo no disponible en caché
else:
    print("Entrenando BERTopic (inglés)…")
    docs_en = english_sample["text"].tolist()

    topic_model_en = BERTopic(
        language="english",
        calculate_probabilities=False,
        verbose=True,
        min_topic_size=15,
        nr_topics="auto",
    )
    topics_en, _ = topic_model_en.fit_transform(docs_en)

    english_sample["topic_id"] = topics_en
    topic_info_en = topic_model_en.get_topic_info()
    print(f"\nTopics encontrados: {len(topic_info_en) - 1} (excl. outliers -1)")

    # Guardar asignaciones
    topic_en_df = english_sample[["post_id", "forum", "userid", "topic_id"]].copy()
    topic_en_df.to_parquet(TOPIC_EN_OUT, index=False)
    print(f"Guardado: {TOPIC_EN_OUT}")


In [ ]:
if topic_model_en is not None:
    topic_info_en = topic_model_en.get_topic_info()
    # Excluir outliers (-1)
    top_topics = topic_info_en[topic_info_en["Topic"] != -1].head(20)
    print("Top 20 topics (foros en inglés):")
    print(top_topics[["Topic", "Count", "Name"]].to_string(index=False))
else:
    # Leer info del parquet cacheado
    topic_counts = (
        topic_en_df[topic_en_df["topic_id"] != -1]
        .groupby("topic_id")
        .size()
        .reset_index(name="Count")
        .sort_values("Count", ascending=False)
        .head(20)
    )
    print("Top 20 topics por tamaño (desde caché):")
    print(topic_counts.to_string(index=False))


In [ ]:
if topic_model_en is not None:
    topic_info_plot = topic_model_en.get_topic_info()
    top15 = topic_info_plot[topic_info_plot["Topic"] != -1].head(15)
    labels = top15["Name"].tolist()
    counts = top15["Count"].tolist()
else:
    topic_counts_plot = (
        topic_en_df[topic_en_df["topic_id"] != -1]
        .groupby("topic_id")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(15)
    )
    labels = [f"Topic {tid}" for tid in topic_counts_plot["topic_id"]]
    counts = topic_counts_plot["count"].tolist()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels[::-1], counts[::-1], color="#4E9EE9", alpha=0.85)
ax.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
ax.set_title("Top 15 topics — foros en inglés (BERTopic)")
ax.set_xlabel("Posts asignados")
plt.tight_layout()
plt.savefig(HF_RESULTS / "topic_en_top15.png", dpi=150, bbox_inches="tight")
plt.show()


---
### Topic modeling — Exploit.in (ruso)

Exploit.in requiere un modelo de embeddings multilingüe.
`sentence-transformers` usa `paraphrase-multilingual-mpnet-base-v2` por defecto,
que cubre ruso con buena calidad. Los topics resultantes aparecen en ruso tal como
están en los posts — no se traducen.

In [ ]:
TOPIC_RU_OUT = HF_RESULTS / "topic_hf_ru_results.parquet"

exploit_posts = posts[posts["lang_route"] == "ru"].copy()
print(f"Posts Exploit.in: {len(exploit_posts):,}")

if len(exploit_posts) == 0:
    print("Sin posts de Exploit.in en posts_clean.parquet — saltando.")
elif TOPIC_RU_OUT.exists():
    topic_ru_df = pd.read_parquet(TOPIC_RU_OUT)
    print(f"Cargado desde caché: {len(topic_ru_df):,} registros")
    topic_model_ru = None
else:
    # Muestrear hasta 5K posts más largos
    exploit_sample = (
        exploit_posts.assign(text_len=exploit_posts["text"].str.len())
        .nlargest(5_000, "text_len")
        .reset_index(drop=True)
    )
    docs_ru = exploit_sample["text"].tolist()
    print(f"Sample: {len(docs_ru):,} posts")

    # Modelo multilingüe explícito
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

    topic_model_ru = BERTopic(
        embedding_model=embedding_model,
        calculate_probabilities=False,
        verbose=True,
        min_topic_size=10,
        nr_topics="auto",
    )
    topics_ru, _ = topic_model_ru.fit_transform(docs_ru)

    exploit_sample["topic_id"] = topics_ru
    topic_ru_info = topic_model_ru.get_topic_info()
    print(f"\nTopics encontrados: {len(topic_ru_info) - 1} (excl. outliers)")
    print("\nTop 10 topics Exploit.in (ruso):")
    print(topic_ru_info[topic_ru_info["Topic"] != -1].head(10)[["Topic", "Count", "Name"]].to_string(index=False))

    topic_ru_df = exploit_sample[["post_id", "forum", "userid", "topic_id"]].copy()
    topic_ru_df.to_parquet(TOPIC_RU_OUT, index=False)
    print(f"\nGuardado: {TOPIC_RU_OUT}")


---
## Sección 3 — NER con qwen2.5:14b (foros en inglés)

Extraemos entidades relevantes al contexto de hacking underground:

| Tipo | Descripción |
|---|---|
| **PERSON** | Nombres reales mencionados |
| **ORGANIZATION** | Empresas, agencias, servicios |
| **ALIAS** | Handles/nicks online |
| **TOOL** | Herramientas de hacking (Metasploit, SQLMap…) |
| **MALWARE** | Nombres de malware (RAT, ransomware, stealer…) |
| **CVE** | Identificadores de vulnerabilidades (CVE-XXXX-XXXXX) |
| **MARKETPLACE** | Mercados darkweb (Genesis Market, RussianMarket…) |

Pipeline:
1. Seleccionar los **top 50 usuarios** por nº de posts (excluyendo Exploit.in)
2. Para cada usuario: tomar hasta **50 posts más largos**
3. Llamar a `qwen2.5:14b` vía ollama con un prompt estructurado
4. Checkpoint cada 10 usuarios

In [ ]:
NER_OUT = HF_RESULTS / "ner_hf_results.parquet"
NER_CHECKPOINT = HF_RESULTS / "ner_hf_checkpoint.json"

TOP_N_USERS = 50
MAX_POSTS_PER_USER = 50

# Top 50 usuarios por posts (solo foros en inglés)
user_post_counts = (
    posts[posts["lang_route"] == "en"]
    .groupby(["forum", "userid"])
    .size()
    .reset_index(name="n_posts")
    .sort_values("n_posts", ascending=False)
)
top_users = user_post_counts.head(TOP_N_USERS).reset_index(drop=True)
print(f"Top {TOP_N_USERS} usuarios seleccionados:")
print(top_users.to_string(index=False))


In [ ]:
NER_SYSTEM = """You are a forensic NER system specialized in underground hacking forum posts.
Extract named entities and return ONLY a valid JSON array. No explanation, no markdown.

Entity types:
- PERSON: real human names
- ORGANIZATION: companies, agencies, services, platforms
- ALIAS: online handles, usernames, nicknames
- TOOL: hacking tools, exploit frameworks, software used for attacks
- MALWARE: malware families, ransomware, stealers, RATs, botnets
- CVE: vulnerability identifiers (format: CVE-YYYY-NNNNN)
- MARKETPLACE: darkweb markets, cybercrime forums, carding shops

Return format: [{"entity": "name", "type": "TYPE"}, ...]
If no entities found, return: []"""

def extract_entities(text: str, model: str = "qwen2.5:14b") -> list[dict]:
    """Run NER on a single text chunk via ollama."""
    prompt = f"Extract entities from this hacking forum post:\n\n{text[:2000]}"
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": NER_SYSTEM},
                {"role": "user", "content": prompt},
            ],
            options={"temperature": 0.0},
        )
        raw = response["message"]["content"].strip()
        # Robust JSON extraction
        start = raw.find("[")
        end = raw.rfind("]") + 1
        if start == -1 or end == 0:
            return []
        return json.loads(raw[start:end])
    except (json.JSONDecodeError, KeyError, Exception):
        return []

# Smoke test
sample_text = "They used CVE-2021-44228 (Log4Shell) with Cobalt Strike to hit the target."
test_entities = extract_entities(sample_text)
print("Smoke test:")
print(json.dumps(test_entities, indent=2))


In [ ]:
CHECKPOINT_EVERY = 10

# Cargar checkpoint si existe
if NER_CHECKPOINT.exists():
    with open(NER_CHECKPOINT) as f:
        ner_checkpoint = json.load(f)
    print(f"Checkpoint cargado: {len(ner_checkpoint)} usuarios procesados")
else:
    ner_checkpoint = {}

ner_records = []

for _, row in tqdm(top_users.iterrows(), total=len(top_users), desc="NER usuarios"):
    forum = row["forum"]
    userid = str(row["userid"])
    user_key = f"{forum}::{userid}"

    if user_key in ner_checkpoint:
        # Recuperar desde checkpoint
        ner_records.extend(ner_checkpoint[user_key])
        continue

    # Posts del usuario — los más largos
    user_posts = (
        posts[
            (posts["forum"] == forum) &
            (posts["userid"].astype(str) == userid) &
            (posts["lang_route"] == "en")
        ]
        .assign(text_len=lambda df: df["text"].str.len())
        .nlargest(MAX_POSTS_PER_USER, "text_len")
    )

    user_entities = []
    for _, post_row in user_posts.iterrows():
        entities = extract_entities(post_row["text"])
        for ent in entities:
            if isinstance(ent, dict) and "entity" in ent and "type" in ent:
                user_entities.append({
                    "forum": forum,
                    "userid": userid,
                    "post_id": post_row.get("post_id", ""),
                    "entity": ent["entity"],
                    "entity_type": ent["type"],
                })

    ner_checkpoint[user_key] = user_entities
    ner_records.extend(user_entities)

    # Checkpoint cada N usuarios
    processed = sum(1 for k in top_users.itertuples()
                    if f"{k.forum}::{k.userid}" in ner_checkpoint)
    if processed % CHECKPOINT_EVERY == 0:
        with open(NER_CHECKPOINT, "w") as f:
            json.dump(ner_checkpoint, f, ensure_ascii=False)
        print(f"  Checkpoint guardado ({processed} usuarios)")

# Checkpoint final
with open(NER_CHECKPOINT, "w") as f:
    json.dump(ner_checkpoint, f, ensure_ascii=False)

ner_df = pd.DataFrame(ner_records)
print(f"\nEntidades extraídas: {len(ner_df):,}")
print(f"Tipos: {ner_df['entity_type'].value_counts().to_dict() if len(ner_df) > 0 else {}}")


In [ ]:
if len(ner_df) == 0:
    print("Sin entidades para visualizar.")
else:
    # ── Distribución por tipo ───────────────────────────────────────────────
    type_counts = ner_df["entity_type"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(type_counts.index[::-1], type_counts.values[::-1],
                 color="#E94E4E", alpha=0.85)
    axes[0].set_title("Distribución de tipos de entidad (NER)")
    axes[0].set_xlabel("Menciones")
    for i, v in enumerate(type_counts.values[::-1]):
        axes[0].text(v + 0.5, i, str(v), va="center", fontsize=8)

    # ── Top entidades por tipo más frecuente ────────────────────────────────
    top_type = type_counts.index[0]
    top_entities = (
        ner_df[ner_df["entity_type"] == top_type]["entity"]
        .str.lower()
        .value_counts()
        .head(15)
    )
    axes[1].barh(top_entities.index[::-1], top_entities.values[::-1],
                 color="#4E9EE9", alpha=0.85)
    axes[1].set_title(f"Top 15 entidades — {top_type}")
    axes[1].set_xlabel("Menciones")

    plt.tight_layout()
    plt.savefig(HF_RESULTS / "ner_en_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Top 10 entidades por cada tipo ──────────────────────────────────────
    print("\n── Top 10 entidades por tipo ──")
    for etype in ["TOOL", "MALWARE", "CVE", "MARKETPLACE", "ALIAS", "ORGANIZATION", "PERSON"]:
        sub = ner_df[ner_df["entity_type"] == etype]["entity"].value_counts().head(10)
        if len(sub) > 0:
            print(f"\n{etype}:")
            print(sub.to_string())


---
### NER — Exploit.in: nota metodológica

**Exploit.in queda excluido del pipeline NER anterior** por las siguientes razones:

1. **Modelos de idioma**: `qwen2.5:14b` fue entrenado principalmente en inglés y chino.
   Su rendimiento en NER ruso es significativamente inferior — comete errores de
   reconocimiento en nombres propios y tiende a transliterar entidades en lugar de
   extraerlas correctamente.

2. **Modelos rusos recomendados**: Para NER en ruso, los modelos apropiados son:
   - `DeepPavlov/rubert-base-cased-sentence` (BERT fine-tuned para NER en ruso)
   - `ai-forever/ruRoberta-large` con adaptador NER
   - `spacy` con el modelo `ru_core_news_lg`

3. **Implicación forense**: Los actores de Exploit.in operan principalmente en ruso.
   Extraer aliases, herramientas y malware de esos posts requiere un pipeline
   separado con los modelos indicados — fuera del alcance de este notebook.

4. **Alternativa práctica**: Las entidades en formato estructurado (CVEs, URLs,
   nombres de malware en inglés) sí pueden extraerse con regex sobre los posts de
   Exploit.in independientemente del idioma, complementando el análisis.

---
## Sección 4 — Guardar resultados

Consolidamos los tres artefactos principales del análisis LLM.

In [ ]:
# ── NER results ─────────────────────────────────────────────────────────────
if len(ner_df) > 0:
    ner_df.to_parquet(NER_OUT, index=False)
    print(f"NER guardado:       {NER_OUT}  ({len(ner_df):,} entidades)")
else:
    print("NER: sin entidades para guardar (run completo pendiente).")

# ── Topic assignments ────────────────────────────────────────────────────────
if "topic_en_df" in dir() and len(topic_en_df) > 0:
    # Ya guardado en Cell 8, confirmar
    print(f"Topics EN guardado: {TOPIC_EN_OUT}  ({len(topic_en_df):,} posts)")
else:
    print("Topics EN: caché no disponible.")

if "topic_ru_df" in dir() and len(topic_ru_df) > 0:
    print(f"Topics RU guardado: {TOPIC_RU_OUT}  ({len(topic_ru_df):,} posts)")
else:
    print("Topics RU: caché no disponible.")

# ── Language detection ───────────────────────────────────────────────────────
# Ya guardado en Cell 4, confirmar
print(f"Lang detection:     {LANG_OUT}  ({len(lang_df):,} registros)")

print("\n── Resumen de artefactos ──")
artifacts = [
    ("ner_hf_results.parquet",       NER_OUT),
    ("topic_hf_results.parquet",     TOPIC_EN_OUT),
    ("topic_hf_ru_results.parquet",  TOPIC_RU_OUT),
    ("language_detection.parquet",   LANG_OUT),
]
for name, path in artifacts:
    status = "✓" if path.exists() else "✗ pendiente"
    size = f"{path.stat().st_size / 1024:.1f} KB" if path.exists() else ""
    print(f"  {status}  {name:<35} {size}")
